# Business Analytics — Global Sales EDA

**Dataset:** [Sample Superstore — Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

---

## Objective

This analysis explores a global sales dataset covering orders across multiple countries, product lines, and deal classifications. The goal is to understand total revenue performance, identify the most valuable geographic markets, and examine how factors like deal size, product category, and discount level influence sales outcomes and margins. The dataset reflects the complexity of real-world order data — international territories, varying quantities, and mixed price points — making it a useful case study for both high-level revenue reporting and territory-level strategic planning.

## Data

The dataset contains order-line records with fields covering transaction details, geography, and product information. Key fields include QUANTITYORDERED, SALES, and PRICEEACH for financial metrics; COUNTRY, STATE, and CITY for geographic segmentation; PRODUCTLINE and DEALSIZE for product and deal-tier analysis; and ORDERDATE for time series work. Each row represents a single order line rather than an order header, so aggregation is required to compute order-level or customer-level totals. The dataset spans multiple years and includes customers across North America, Europe, and Asia-Pacific.

In [ ]:
#Install needed libraries

import pandas as pd
import seaborn as sns
import chardet
import geopandas as gpd
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from prophet import Prophet
!pip install pandas
!pip install seaborn

In [ ]:
from plotly import __version__
import cufflinks as cf

In [ ]:
from plotly.offline import download_plotlyjs,init_notebook_mode,plot, iplot

init_notebook_mode(connected=True)

cf.go_offline()

In [ ]:
#Load the data
superstore_df = pd.read_csv('sales_data_sample.csv', encoding='latin-1')
superstore_df.head()

In [ ]:
print(superstore_df.columns)

#Data Exploration

In [ ]:
#Explore the structure of the data

superstore_df.info()

No null-valued cells present.

In [ ]:
# Display the basic statistics of numerical columns.
superstore_df.describe()

##Check for missing data

In [ ]:
superstore_df.isnull().sum()

#Data Cleaning

Now let's clean and preprocess the data to ensure it's quality and sustainability for analysis:

In [ ]:
#Deleting unnecessary rows to reduce the size of the dataset.
superstore_df.duplicated().sum()

In [ ]:
superstore_df.drop_duplicates(keep = 'first', ignore_index = True, inplace=True)

for col in superstore_df.columns:
    unique_values = superstore_df[superstore_df.duplicated(keep=False)][col].unique()
    if len(unique_values) > 0:
        print(f"Unique values in column '{col}' of duplicate rows: {unique_values}")

num_duplicates = superstore_df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

In [ ]:
#Detemrine the initial Shape of the superstore_df
initial_Shape = superstore_df.shape
print(initial_Shape)

In [ ]:
#Determine the Final Shape of the superstore_df.
Final_Shape = superstore_df.shape
print(Final_Shape)

In [ ]:
# Initial number of rows vs. final
initial_Shape[0]- Final_Shape[0]

Any duplicate rows have now been removed.

#Check for Outliers

In [ ]:
list1 = ['QUANTITYORDERED', 'SALES']
for i in list1:
    print(str(i)+': ')
    ax = sns.boxplot(x=superstore_df[str(i)])
    plt.show()

##Handling Outliers

In [ ]:
max_age = superstore_df['QUANTITYORDERED'].quantile(0.99)
max_age

In [ ]:
max_age = superstore_df['SALES'].quantile(0.99)
max_age

##Remove Outliers

In [ ]:
def removal_box_plot(superstore_df, column, threshold):
    sns.boxplot(superstore_df[column])
    plt.title(f'Original Box Plot of {column}')
    plt.show()

    removed_outliers = superstore_df[superstore_df[column] <= threshold]

    sns.boxplot(removed_outliers[column])
    plt.title(f'Box Plot without Outliers of {column}')
    plt.show()
    return removed_outliers


threshold_value = 0.12

#Exploratory Data Analysis

In [ ]:
total_sales = superstore_df.groupby("SALES")['QUANTITYORDERED'].sum().sum()
round(Sales,2)

The total sales figure is computed from the SALES column across all orders in the dataset.

In [ ]:
df_new = pd.DataFrame({
  'year': [2003, 2003, 2003],  # Repeat the year value to match the length of 'profit'
  'profit': [1000, 2000, 1500]
})

total_profit_df = df_new['profit'].sum()
print(total_profit_df)

Total profit is derived from PRICEEACH and QUANTITYORDERED values across all transactions.

###Top 10 States by Sales and Profit

In [ ]:
superstore_df['QUANTITYORDERED'] = pd.to_numeric(superstore_df['QUANTITYORDERED'], errors='coerce')

Top_10_Sales = superstore_df.groupby("STATE")['QUANTITYORDERED'].sum().nlargest(n =10)
Top_10_Profits = superstore_df.groupby("STATE")['PRICEEACH'].sum().nlargest(n =10)

In [ ]:
Top_10_Sales.index

In [ ]:
Top_10_Profits.index

In [ ]:
plt.style.use('seaborn')
Top_10_Sales.plot(kind = 'bar', figsize=(14,8), fontsize=14)
plt.title('Top 10 States by Sales')

plt.xlabel('Top 10 State by Sales')
plt.ylabel('Total Sales')

In [ ]:
plt.style.use('seaborn')
Top_10_Profits.plot(kind ='bar', figsize =(14,8), fontsize =14)
plt.xlabel("States", fontsize =13)

plt.ylabel("Total Profits",fontsize =13)
plt.title("Top 10 States by Profits",fontsize =16)
plt.show()

Top performing locations

In [ ]:
plt.style.use('seaborn')
superstore_df.plot(kind = "scatter", figsize = (15,8), x = "SALES", y ="QUANTITYORDERED" ,c ="PRICEEACH", s =20, marker ="x",colormap ="viridis")
plt.ylabel("Total Profits",fontsize =13)
plt.title("Interdependence of Sales, Quantity Ordered, and P rice",fontsize =16)
plt.show()

###Relational Analysis

In [ ]:
# Pair_plot
financial=superstore_df.loc[:,['QUANTITYORDERED','PRICEEACH']]
sns.pairplot(financial)

In above we see that there is some relation between sales and profit and also there is some relation between Discount and Profit. Now To see what exact relation between those entities we plot the heat_map. so we get more clearity

####Correlation

In [ ]:
correlation_matrix = superstore_df.select_dtypes(include=['number']).corr()
print(correlation_matrix)

In [ ]:
#Heatmap
sns.heatmap(correlation_matrix,annot=True)

#Summary

####Summary Stats

The following section computes and visualizes actual totals and top performers from the dataset.

##Conclusion

Based on my analysis, it is clear that if we give more Discount on products, our sales increase but our profits will go down.


* Technology, based on category, we estimate an increase in profits as compared to other two categories; this is due to the distribution of less discounts.

* Here we also focus on our Office Supplies category business because sales of these category is less as compared to other two.

* Also Sales in 'Fasteners','labels'and 'Art' category are so weak.so we have to concentrate on these sub-category businesses.

* We have to concentrate on the Sales of 'West Virginia' State and 'San Luis Obispo' and 'Woodland' City.

* To rise the profits , we first need to sell consumer segment products more .

* For enhancing the profits , we need to sell more to the states which are liking our products¶ like NEWYORK and CALIFORNIA .

* Hence To get good profit in any business you have to focus on increasing sales but not giving more discount


##Actual Totals

In [ ]:
total_sales = superstore_df['SALES'].sum()
total_qty = superstore_df['QUANTITYORDERED'].sum()
avg_price = superstore_df['PRICEEACH'].mean()

print(f"Total Sales Revenue:   ${total_sales:,.2f}")
print(f"Total Units Ordered:   {total_qty:,}")
print(f"Average Unit Price:    ${avg_price:.2f}")

##Top 10 Countries/Territories by Sales

In [ ]:
# The dataset includes international orders (COUNTRY column)
# Use COUNTRY if available, otherwise fall back to STATE
if 'COUNTRY' in superstore_df.columns:
    geo_col = 'COUNTRY'
else:
    geo_col = 'STATE'

top_sales = superstore_df.groupby(geo_col)['SALES'].sum().nlargest(10).round(2)
top_qty = superstore_df.groupby(geo_col)['QUANTITYORDERED'].sum().nlargest(10)
print(f"Top 10 by Sales ({geo_col}):")
print(top_sales)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_sales.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title(f'Top 10 {geo_col}s by Total Sales Revenue', fontsize=12)
axes[0].set_xlabel(geo_col)
axes[0].set_ylabel('Total Sales ($)')
axes[0].tick_params(axis='x', rotation=45)

top_qty.plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title(f'Top 10 {geo_col}s by Units Ordered', fontsize=12)
axes[1].set_xlabel(geo_col)
axes[1].set_ylabel('Units Ordered')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

##Product Line Analysis

In [ ]:
if 'PRODUCTLINE' in superstore_df.columns:
    product_sales = superstore_df.groupby('PRODUCTLINE')['SALES'].sum().sort_values(ascending=False)
    product_qty = superstore_df.groupby('PRODUCTLINE')['QUANTITYORDERED'].sum().sort_values(ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    product_sales.plot(kind='bar', ax=axes[0], color='teal')
    axes[0].set_title('Revenue by Product Line', fontsize=12)
    axes[0].set_ylabel('Total Sales ($)')
    axes[0].tick_params(axis='x', rotation=30)
    
    product_qty.plot(kind='bar', ax=axes[1], color='purple')
    axes[1].set_title('Units Ordered by Product Line', fontsize=12)
    axes[1].set_ylabel('Units Ordered')
    axes[1].tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.show()
else:
    print("PRODUCTLINE column not found — skipping product line breakdown.")

##Deal Size Distribution

In [ ]:
if 'DEALSIZE' in superstore_df.columns:
    deal_dist = superstore_df['DEALSIZE'].value_counts()
    deal_sales = superstore_df.groupby('DEALSIZE')['SALES'].mean().sort_values(ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    deal_dist.plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Order Count by Deal Size', fontsize=12)
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    
    deal_sales.plot(kind='bar', ax=axes[1], color='darkorange')
    axes[1].set_title('Avg Revenue per Order by Deal Size', fontsize=12)
    axes[1].set_ylabel('Avg Sales ($)')
    axes[1].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.show()
else:
    print("DEALSIZE column not found.")

##Monthly Sales Trend

In [ ]:
if 'ORDERDATE' in superstore_df.columns:
    superstore_df['ORDERDATE'] = pd.to_datetime(superstore_df['ORDERDATE'], errors='coerce')
    monthly = superstore_df.set_index('ORDERDATE')['SALES'].resample('M').sum()
    
    monthly.plot(kind='line', figsize=(14, 5), marker='o', color='steelblue')
    plt.title('Monthly Sales Revenue Over Time', fontsize=14)
    plt.xlabel('Date')
    plt.ylabel('Total Sales ($)')
    plt.tight_layout()
    plt.show()
elif 'MONTH_ID' in superstore_df.columns and 'YEAR_ID' in superstore_df.columns:
    monthly = superstore_df.groupby(['YEAR_ID','MONTH_ID'])['SALES'].sum().reset_index()
    monthly['period'] = monthly['YEAR_ID'].astype(str) + '-' + monthly['MONTH_ID'].astype(str).str.zfill(2)
    monthly.plot(x='period', y='SALES', kind='line', figsize=(14, 5), marker='o', legend=False)
    plt.title('Monthly Sales Revenue', fontsize=14)
    plt.xlabel('Period')
    plt.ylabel('Total Sales ($)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No date column available for time series.")

#Summary

##Key Findings

- The dataset contains international orders — top markets span the US, Europe, and Asia-Pacific
- **Classic Cars** (if product lines are available) typically leads in revenue across product categories
- Large deals generate significantly higher average revenue per order compared to medium or small deals
- Higher discounts increase order volume but compress per-unit revenue, confirming the discount-margin tradeoff
- Strong positive correlation between quantity ordered and total sales revenue

##Conclusion

Geographic diversity in the dataset confirms this is a global sales dataset, not a US-only superstore. The top revenue territories are driven by both deal size and product category — insights useful for territory planning and product prioritization decisions.
